# 🔄 Notebook 04: RAG Pipeline & Chat

Notebook này thực hiện:
1. Khởi tạo RAG pipeline
2. Test retrieval + reranking
3. Test chat với multi-turn conversation
4. Test edge cases (out-of-scope, ambiguity, topic change)

In [ ]:
import os, sys
PROJECT_DIR = '/content/vietnamese-legal-qa'
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

from src.config.settings import Settings
from src.components.embedding_engine import EmbeddingEngine
from src.components.vector_store import VectorStoreManager
from src.components.retriever import Retriever
from src.components.conversation_manager import ConversationManager
from src.components.llm_engine import LLMEngine
from src.components.quality_guard import QualityGuard
from src.services.chat_service import ChatService

settings = Settings.load('config.yaml')
print('Config loaded!')

In [ ]:
# === Khởi tạo components ===
embedding_engine = EmbeddingEngine(settings)
vector_store = VectorStoreManager(settings)
retriever = Retriever(settings, embedding_engine, vector_store)
conversation_manager = ConversationManager(settings)
llm_engine = LLMEngine(settings)
quality_guard = QualityGuard(settings)

# Load fine-tuned model
llm_engine.load_model('qwen', adapter_path='models/adapters/qwen')

# Khởi tạo ChatService
chat_service = ChatService(
    settings=settings,
    retriever=retriever,
    conversation_manager=conversation_manager,
    llm_engine=llm_engine,
    quality_guard=quality_guard,
)

print('✅ All components initialized!')

In [ ]:
# === Test Chat: Basic Q&A ===
session_id = chat_service.new_session()

response = chat_service.chat(
    user_message="Điều kiện kết hôn theo pháp luật Việt Nam là gì?",
    session_id=session_id,
    model_key='qwen'
)

print(f'Answer: {response.answer}')
print(f'\nSources ({len(response.sources)}):')
for s in response.sources:
    print(f'  - {s.breadcrumb} [score: {s.relevance_score:.3f}]')
print(f'\nSuggestions: {response.suggestions}')
print(f'Time: {response.inference_time:.2f}s')

In [ ]:
# === Test Multi-turn Conversation (US-11) ===
print('=== Multi-turn Test ===')

# Turn 1
r1 = chat_service.chat("Thời hạn hợp đồng lao động xác định?", session_id, 'qwen')
print(f'Q1: Thời hạn hợp đồng lao động xác định?')
print(f'A1: {r1.answer[:200]}...\n')

# Turn 2 - Reference resolution
r2 = chat_service.chat("Vậy nếu hết hạn mà không ký lại thì sao?", session_id, 'qwen')
print(f'Q2: Vậy nếu hết hạn mà không ký lại thì sao?')
print(f'A2: {r2.answer[:200]}...\n')

# Turn 3 - Still same topic
r3 = chat_service.chat("Người lao động có quyền đơn phương chấm dứt không?", session_id, 'qwen')
print(f'Q3: Người lao động có quyền đơn phương chấm dứt không?')
print(f'A3: {r3.answer[:200]}...')

In [ ]:
# === Test Topic Change (US-12) ===
print('=== Topic Change Test ===')

r4 = chat_service.chat("Còn nếu tôi muốn ly hôn thì thủ tục thế nào?", session_id, 'qwen')
print(f'Q4: Còn nếu tôi muốn ly hôn thì thủ tục thế nào?')
print(f'A4: {r4.answer[:200]}...')
print(f'\n(Topic should have changed from labor to family law)')

In [ ]:
# === Test Out-of-Scope (US-14) ===
print('=== Out-of-Scope Test ===')

r_oos = chat_service.chat("Cho tôi công thức nấu phở", session_id, 'qwen')
print(f'Q: Cho tôi công thức nấu phở')
print(f'Rejected: {r_oos.is_rejection}')
print(f'A: {r_oos.answer}')

In [ ]:
# === Test Ambiguity (US-16) ===
print('=== Ambiguity Test ===')

r_amb = chat_service.chat("Luật hợp đồng?", session_id, 'qwen')
print(f'Q: Luật hợp đồng?')
print(f'Ambiguous: {r_amb.is_clarification}')
print(f'Options: {r_amb.clarification_options}')
print(f'A: {r_amb.answer}')

In [ ]:
# === Test Session Summary (US-19) ===
summary = chat_service.get_session_summary(session_id)
print('=== Session Summary ===')
print(summary)